# 第 3 章 · 执行环境与上下文管理

> 对应文档：**Agents → Configure the harness → Execution environment / Context management**
> https://docs.langchain.com/oss/python/langchain/agents#configure-the-harness
>
> 每个小节的结构都一样：**文档怎么说 → 平台上对应哪里 → 练习怎么做 → 看什么算证据**。
> Python 块是文档里的原写法，**只作对照，不用运行** —— 在这个平台上，同样的东西是
> 在 agent 编辑器的五个页签里填字段：Overview / Prompt / Tools / Skills / JSON。


## 这一章在讲什么

> **Execution environment** — Agents are especially useful when they can take action rather
> than just generate text. The execution environment gives the agent a workspace: tools it
> can call, a filesystem for reading and writing files across turns, and code execution.
>
> **Context management** — Every model call has a fixed context window. As an agent runs,
> that window fills with history, tool results, intermediate steps. Summarization compresses
> history before overflow; memory loads persistent instructions at startup; **skills surface
> domain knowledge on demand**.

先记住一个判断：**这两件事解决的是同一个问题 —— 模型那一格窗口装不下。**
执行环境把东西**放出去**（文件、脚本），上下文管理把东西**压下去**（摘要、记忆、技能）。

## 1 · Execution environment（工作区、文件、代码）

```python
from deepagents.backends import StateBackend
from deepagents.middleware import FilesystemMiddleware
agent = create_agent(model=..., tools=[search],
                     middleware=[FilesystemMiddleware(backend=StateBackend())])
```

**平台上对应哪里**：**默认就有**，不需要配。每个 agent 有自己的工作区：

```
outputs/users/<账号 id>/<vault>/agents/agent_<id>/
    upload/      你上传的文件
    generated/   它产出的成品
    other/       草稿
    skills/      这个 agent 的技能
```

**练习**

1. 上传一个文件，然后问 `我上传了什么？`
2. 让它写一份短报告，再看 **Files 面板**里的 `generated/`。

**看什么算证据**

- 判断"它产出了东西"，看**文件是否存在**，不看聊天里说"我已经保存"。
- 这套工作区是**按账号隔离**的：别人登录时看到的是他自己那棵树。

> 这一章是本平台比文档的样例更强的地方：文档示例里 `StateBackend()` 是进程内状态，
> 平台给的是**磁盘上的真实目录 + 按账号隔离**。这也是唯一一类"换个 prompt 绝不会解决"
> 的失败：没有文件工具，写到哪都是假的。

## 2 · Context management（摘要、记忆、技能）

```python
middleware=[
    SummarizationMiddleware(model=model, backend=backend),
    MemoryMiddleware(backend=backend, sources=["./AGENTS.md"]),
    SkillsMiddleware(backend=backend, sources=["./skills/"]),
]
```

**平台上对应哪里**

| 文档里的中间件 | 平台上 |
|---|---|
| `SummarizationMiddleware` | **默认开启**：guardrails 里的 token 上限/淘汰阈值/摘要阈值（Overview 的高级设置） |
| `MemoryMiddleware` | **默认开启**：`AGENTS.md` 每轮内联进 prompt |
| `SkillsMiddleware` | **Skills 页签**：勾选技能，agent 按需读取 |

**练习（技能这一条最值得做）**

1. 给 agent 勾一个技能（例如 `sql-analyst`），问一个**不相关**的问题 → 它应该**不读**技能。
2. 再问一个**相关**的问题 → 它应该先调用"读技能"，再回答。

**看什么算证据**：技能不是"塞进 prompt 的一大段文字"，而是**一个可被读的工具**。
判断依据是**读技能的那次调用**有没有出现；出现得早，说明描述写得准；不出现，
说明技能描述没能让模型相信"这时候该翻这本书"。

**为什么技能属于这一章**：它就是"把上下文省下来"的手段 —— 不用时不占窗口。
这也是为什么技能不该被当成"能力"来讲：它不增加 agent 能做的事，只改变**什么时候知道**。

## 3 · 什么时候这一章会失效（要讲清的边界）

| 现象 | 真正原因 | 改 prompt 有用吗 |
|---|---|---|
| 大文件答得越来越糊 | 摘要把早期内容压掉了 | ❌ 没用 |
| 上传了文件但它说没有 | 上传 ≠ 已索引（检索型流程需要索引步骤） | ❌ 没用 |
| 技能不触发 | 技能描述不匹配问题 | ⚠️ 改描述有用，改正文没用 |

**这一章的结论**：上下文管理不是"prompt 写得更好"，是**在窗口之外给它一套存取机制**。